In [11]:
from selenium import webdriver
from bs4 import BeautifulSoup
import time
from selenium.common.exceptions import TimeoutException


driver = webdriver.Chrome()
search_url ='https://www.daraz.pk/catalog/?q=smartphones'
driver.set_page_load_timeout(30)
try:
    driver.get(search_url)
    time.sleep(2)
    html_content = driver.page_source
    
    
    with open("mobile.html","w",encoding='utf-8') as f:
         f.write(html_content)
         
    print(f"page content save in html file succesffuly")     
    
except TimeoutException:
        print(f'time out check internet connection')   
finally:     
   html_content = driver.page_source
  
driver.quit()       

page content save in html file succesffuly


In [12]:
# check product len
len(html_content)

1053593

In [3]:
from bs4 import BeautifulSoup
with open('data/raw/mobile.html',"r",encoding='utf-8') as f:
    soup= BeautifulSoup(f,'html.parser')
    
product_links = []

for links in soup.find_all("a",href=True):
    href = links["href"]
    
    if href and "/products" in href:
        product_links.append(href)
        
product_links =list(dict.fromkeys(product_links))
    
# Print the total number of unique product links found on the page
print(f'Total Unique Products Links: {len(product_links)}')

Total Unique Products Links: 40


In [4]:
with open('data/raw/mobile.html',"r") as f:
    for i, link in enumerate(product_links, start=1):
        print(i, link)
f.close()

1 //www.daraz.pk/products/apple-iphone-17-pro-max-512gb-i1957775263.html
2 //www.daraz.pk/products/apple-iphone-17-pro-max-256gb-i1958136639.html
3 //www.daraz.pk/products/redmi-15c-4gb-128gb-pta-approved-i923138689.html
4 //www.daraz.pk/products/infinix-smart-20-4gb-64gb-i1958170417.html
5 //www.daraz.pk/products/realme-c100i-4gb64gb-i1958905393.html
6 //www.daraz.pk/products/vivo-y17-pta-approved-original-mobile-phone-8gb-ram-256gb-rom-5000mah-battery-dual-sim-fingerprint-android-smartphone-global-variant-like-new-box-charger-handsfree-included-installment-available-best-budget-phone-in-pakistan-i1958256534.html
7 //www.daraz.pk/products/vivo-y17-635-ips-lcd-8gb-ram-256gb-pta-helio-p35-i491864707.html
8 //www.daraz.pk/products/apple-iphone-16-128gb-i1958148618.html
9 //www.daraz.pk/products/apple-iphone-17-pro-512gb-i1957827008.html
10 //www.daraz.pk/products/realme-gt-7t-12gb-ram-512gb-rom-pta-approved-i895953652.html
11 //www.daraz.pk/products/y85-4-64-622-i1950844162.html
12 //www

In [8]:
# extract data now
import pandas as pd
from selenium import webdriver
import time
import json
from bs4 import BeautifulSoup
import time 
import re

try:
    driver = webdriver.Chrome()
    
    products=[]
    
    for i ,links in enumerate( product_links , start=1):
        if links.startswith('//'):
            links= 'https:'+links
        elif links.startswith('/'):
            links= "https://www.daraz.pk"+links
        if not links.startswith('http'):
            continue
        driver.get(links)
        time.sleep(2)
        
        html_data = driver.page_source
        
        match =  re.search(
        r'var pdpTrackingData = "(.*?)";',
        html_data,
        re.DOTALL
    )
        
        if match:
            try:
               data_string= match.group(1)
               data_string = data_string.replace('\\"', '"')
               
               data=  json.loads(data_string) 
               product = {
                               "Product_Name": data.get("pdt_name"),
                               "Price": data.get("pdt_price"),
                               "Brand": data.get("brand_name"),
                               "SKU": data.get("pdt_sku"),
                               "Category": data.get("pdt_category"),
                               "Seller": data.get("seller_name"),
                               "Product_URL": links
                           }
               products.append(product)
               
               print(f"{i}/40 - {product['Product_Name']}")
               
            except Exception as e:
                           print(f"{i}/40 - JSON Error:", e)
               
        else:
                       print(f"{i}/40 - Tracking data not found")
               
               
    print("\nScraping completed!")
    print("Products collected:", len(products))
except Exception as e:
    print(f'{e}')
    
    

1/40 - Apple iPhone 17 Pro Max - 512GB
2/40 - Apple iPhone 17 Pro Max - 256GB
3/40 - Redmi 15C 4GB +128GB - PTA APPROVED
4/40 - Infinix Smart 20 4GB / 64GB
5/40 - Realme C100i 4GB / 64GB
6/40 - Vivo Y17 PTA Approved Original Mobile Phone | 8GB RAM 256GB ROM | 5000mAh Battery | Dual SIM | Fingerprint | Android Smartphone | Global Variant | Like New | Box Charger Handsfree Included | Installment Available | Best Budget Phone in Pakistan
7/40 - Vivo Y17 - 6.35 Inches IPS LCD Display - 8GB RAM - 256GB Storage - Official PTA Approved - Fingerprint Sensor - Helio P35 Processor - (Phone and Box Charger, No Accessories Included - Mix Color)
8/40 - Apple iPhone 16 - 128GB
9/40 - Apple iPhone 17 Pro - 512GB
10/40 - Realme GT 7T 12GB RAM 512GB ROM - PTA APPROVED
11/40 - Original Vivo Y85 Smartphone | 4GB 64GB | 6.22-inch Notch Screen | PTA Approved | Snapdragon High Performance | FREE Charger Included | Best Discounted Rate | Installment Plan
12/40 - Apple iPhone 17 Pro - 256GB
13/40 - vivo Y05 4

In [13]:
df = pd.DataFrame(products)
len(df)
df

,Product_Name,Price,Brand,SKU,Category,Seller,Product_URL
0,Apple iPhone 17 Pro Max - 512GB,"Rs. 612,799",Apple,1957775263,[Mobiles & Tablets],,https://www.daraz.pk/products/apple-iphone-17-...
1,Apple iPhone 17 Pro Max - 256GB,"Rs. 535,600",Apple,1958136639,[Mobiles & Tablets],,https://www.daraz.pk/products/apple-iphone-17-...
2,Redmi 15C 4GB +128GB - PTA APPROVED,"Rs. 46,999",Xiaomi Official,923138689,[Mobiles & Tablets],,https://www.daraz.pk/products/redmi-15c-4gb-12...
3,Infinix Smart 20 4GB / 64GB,"Rs. 41,999",Infinix,1958170417,[Mobiles & Tablets],,https://www.daraz.pk/products/infinix-smart-20...
4,Realme C100i 4GB / 64GB,"Rs. 40,999",Realme Official,1958905393,[],,https://www.daraz.pk/products/realme-c100i-4gb...
5,Vivo Y17 PTA Approved Original Mobile Phone | ...,"Rs. 19,799",No Brand,1958256534,[Mobiles & Tablets],,https://www.daraz.pk/products/vivo-y17-pta-app...
6,Vivo Y17 - 6.35 Inches IPS LCD Display - 8GB R...,"Rs. 24,999",No Brand,491864707,[Mobiles & Tablets],,https://www.daraz.pk/products/vivo-y17-635-ips...
7,Apple iPhone 16 - 128GB,"Rs. 345,000",Apple,1958148618,[Mobiles & Tablets],,https://www.daraz.pk/products/apple-iphone-16-...
8,Apple iPhone 17 Pro - 512GB,"Rs. 574,100",Apple,1957827008,[Mobiles & Tablets],,https://www.daraz.pk/products/apple-iphone-17-...
9,Realme GT 7T 12GB RAM 512GB ROM - PTA APPROVED,"Rs. 169,999",Realme Official,895953652,[Mobiles & Tablets],,https://www.daraz.pk/products/realme-gt-7t-12g...


In [14]:
df.to_csv('mobile.csv',index=False,encoding='utf-8')